In [ ]:
# !/bin/bash
!curl -L -o ./tiny-imagenet-200.zip\
  https://image-net.org/data/tiny-imagenet-200.zip

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  236M  100  236M    0     0  46.8M      0  0:00:05  0:00:05 --:--:-- 50.7M


In [ ]:
!pip install patool

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.5/86.5 kB 2.5 MB/s eta 0:00:00


In [ ]:
import patoolib
patoolib.extract_archive("./tiny-imagenet-200.zip", outdir="./")

INFO patool: Extracting ./tiny-imagenet-200.zip ...
INFO:patool:Extracting ./tiny-imagenet-200.zip ...
INFO patool: running /usr/bin/7z x -aou -o./ -- ./tiny-imagenet-200.zip
INFO:patool:running /usr/bin/7z x -aou -o./ -- ./tiny-imagenet-200.zip
INFO patool: ... ./tiny-imagenet-200.zip extracted to `./'.
INFO:patool:... ./tiny-imagenet-200.zip extracted to `./'.


'./'

In [ ]:
import os
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms

class ImageNetTrainDataset(Dataset):
    def __init__(self, root, transform=None):
        self.transform = transform
        self.samples = []
        self.class_to_idx = {}
        train_dir = os.path.join(root, 'train')
        classes = sorted([d for d in os.listdir(train_dir) if d.startswith('n') and os.path.isdir(os.path.join(train_dir, d))])
        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(classes)}
        for cls in classes:
            img_dir = os.path.join(train_dir, cls, 'images')
            if not os.path.isdir(img_dir):
                continue
            for fname in os.listdir(img_dir):
                if fname.lower().endswith(('.jpeg', '.jpg', '.png')):
                    self.samples.append((os.path.join(img_dir, fname), self.class_to_idx[cls]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label


class ImageNetValDataset(Dataset):
    def __init__(self, root, transform=None):
        self.transform = transform
        val_dir = os.path.join(root, 'val')
        self.images_dir = os.path.join(val_dir, 'images')
        self.img_names = sorted([f for f in os.listdir(self.images_dir) if f.lower().endswith(('.jpeg', '.jpg', '.png'))])
        ann_path = os.path.join(val_dir, 'val_annotations.txt')
        self.img_to_class = {}
        with open(ann_path, 'r') as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) >= 2:
                    self.img_to_class[parts[0]] = parts[1]
        class_names = sorted(set(self.img_to_class.values()))
        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(class_names)}
        self.samples = []
        for img_name in self.img_names:
            cls = self.img_to_class.get(img_name)
            if cls is None:
                continue
            label = self.class_to_idx[cls]
            self.samples.append((os.path.join(self.images_dir, img_name), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

class ImageNetTestDataset(Dataset):
    def __init__(self, root, transform=None):
        """
        root - path to ImageNet root (folder that contains 'test/images')
        - test/images/*.JPEG contains test images (without labels)
        """
        self.transform = transform
        test_dir = os.path.join(root, 'test', 'images')
        self.samples = sorted(os.listdir(test_dir))
        self.samples = [os.path.join(test_dir, f) for f in self.samples]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        # No label in test set
        return image, -1  # Or just return image only if you modify Dataloader/use case

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, 1, 1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, 1, 1)
        self.bn2 = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU()

    def forward(self, x):
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += residual
        return self.relu(out)

class PayloadEncoder(nn.Module):
    def __init__(self, payload_bits):
        super().__init__()
        self.payload_bits = payload_bits
        self.cover_analyzer = nn.Sequential(
            nn.Conv2d(3, 64, 3, 1, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 128, 3, 1, 1),
            nn.BatchNorm2d(128),
            nn.ReLU()
        )
        self.secret_expand = nn.Sequential(
            nn.Linear(payload_bits, 512),
            nn.ReLU(),
            nn.Linear(512, 1024),
            nn.ReLU(),
            nn.Linear(1024, 128 * 4 * 4)
        )
        self.embedder = nn.Sequential(
            nn.Conv2d(256, 256, 3, 1, 1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, 128, 3, 1, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 64, 3, 1, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 3, 3, 1, 1),
            nn.Tanh()
        )
        self.embedding_strength = 0.2

    def forward(self, cover, secret):
        B, C, H, W = cover.shape
        cover_feat = self.cover_analyzer(cover)
        secret_exp = self.secret_expand(secret).view(B, 128, 4, 4)
        secret_upsampled = F.interpolate(secret_exp, size=(H, W), mode='bilinear', align_corners=False)
        combined = torch.cat([cover_feat, secret_upsampled], dim=1)
        embedding = self.embedder(combined)
        stego = cover + self.embedding_strength * embedding
        stego = torch.clamp(stego, 0, 1)
        return stego, embedding

class PayloadDecoder(nn.Module):
    def __init__(self, payload_bits):
        super().__init__()
        self.payload_bits = payload_bits
        self.feature_extractor = nn.Sequential(
            nn.Conv2d(3, 64, 3, 1, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, 1, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 128, 3, 2, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, 3, 1, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 256, 3, 2, 1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, 256, 3, 1, 1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(4)
        )
        self.secret_recovery = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, payload_bits)
        )

    def forward(self, stego):
        features = self.feature_extractor(stego)
        secret = self.secret_recovery(features)
        return secret

class RealisticDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
            nn.Conv2d(128, 256, 4, 2, 1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        return self.model(x)

class SteganographySystem(nn.Module):
    def __init__(self, payload_bits=32):
        super().__init__()
        self.encoder = PayloadEncoder(payload_bits)
        self.decoder = PayloadDecoder(payload_bits)
        self.discriminator = RealisticDiscriminator()

    def forward(self, cover, secret):
        stego, embedding = self.encoder(cover, secret)
        extracted = self.decoder(stego)
        return stego, extracted, embedding

In [ ]:
import os
import math
import torch
import torch.nn.functional as F
from torch import nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, DistributedSampler
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP
from tqdm import tqdm

import torch.optim as optim

# def setup_distributed(rank, world_size):
#     dist.init_process_group(
#         backend='nccl',
#         init_method='env://',
#         world_size=world_size,
#         rank=rank)
#     torch.cuda.set_device(rank)

# def cleanup_distributed():
#     dist.destroy_process_group()


def train_one_epoch(epoch, model, dataloader, optimizer_enc, optimizer_dec, optimizer_disc, device):
    model.train()
    metrics = {'total_loss': 0, 'secret_loss': 0, 'image_loss': 0, 'adv_loss': 0, 'embed_reg': 0}
    bit_accs = []
    psnrs = []
    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1} Training")
    for images, _ in pbar:
        images = images.to(device)
        B = images.shape[0]
        secret = torch.randint(0, 2, (B, model.encoder.payload_bits), dtype=torch.float32, device=device)

        if epoch >= 10:
            optimizer_disc.zero_grad()
            with torch.no_grad():
                stego, _, _ = model(images, secret)
            real_pred = model.discriminator(images)
            fake_pred = model.discriminator(stego.detach())
            real_loss = F.binary_cross_entropy_with_logits(real_pred, torch.ones_like(real_pred))
            fake_loss = F.binary_cross_entropy_with_logits(fake_pred, torch.zeros_like(fake_pred))
            disc_loss = 0.5 * (real_loss + fake_loss)
            disc_loss.backward()
            optimizer_disc.step()
        else:
            disc_loss = torch.tensor(0., device=device)

        optimizer_enc.zero_grad()
        optimizer_dec.zero_grad()

        stego, extracted, embedding = model(images, secret)

        secret_loss = F.binary_cross_entropy_with_logits(extracted, secret) * 20
        image_loss = F.mse_loss(stego, images) * 2

        adv_loss = torch.tensor(0., device=device)
        if epoch >= 10:
            fake_pred = model.discriminator(stego)
            adv_loss = F.binary_cross_entropy_with_logits(fake_pred, torch.ones_like(fake_pred)) * 0.5

        embed_reg = torch.mean(torch.abs(embedding)) * 0.1

        total_loss = secret_loss + image_loss + adv_loss + embed_reg
        total_loss.backward()

        torch.nn.utils.clip_grad_norm_(model.encoder.parameters(), 1.0)
        torch.nn.utils.clip_grad_norm_(model.decoder.parameters(), 1.0)

        optimizer_enc.step()
        optimizer_dec.step()

        with torch.no_grad():
            pred_bits = (torch.sigmoid(extracted) > 0.5).float()
            bit_acc = (pred_bits == secret).float().mean().item() * 100
            mse_val = F.mse_loss(stego, images).item()
            psnr = 20 * math.log10(1.0 / math.sqrt(mse_val + 1e-8))

        for k in metrics:
            if k == 'secret_loss':
                metrics[k] += secret_loss.item() * B
            elif k == 'image_loss':
                metrics[k] += image_loss.item() * B
            elif k == 'adv_loss':
                metrics[k] += adv_loss.item() * B
            elif k == 'embed_reg':
                metrics[k] += embed_reg.item() * B
            elif k == 'total_loss':
                metrics[k] += total_loss.item() * B

        bit_accs.append(bit_acc)
        psnrs.append(psnr)

        pbar.set_postfix(BitAcc=f"{bit_acc:.2f}%", PSNR=f"{psnr:.2f}dB", Loss=f"{total_loss.item():.5f}")

    size = len(dataloader.dataset)
    for k in metrics:
        metrics[k] /= size

    avg_acc = sum(bit_accs) / len(bit_accs)
    avg_psnr = sum(psnrs) / len(psnrs)
    return metrics, avg_acc, avg_psnr

def validate(epoch, model, dataloader, device):
    model.eval()
    bit_accs = []
    psnrs = []
    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1} Validation")
    with torch.no_grad():
        for images, _ in pbar:
            images = images.to(device)
            B = images.shape[0]
            secret = torch.randint(0, 2, (B, model.encoder.payload_bits), dtype=torch.float32, device=device)
            stego, extracted, _ = model(images, secret)
            pred_bits = (torch.sigmoid(extracted) > 0.5).float()
            bit_acc = (pred_bits == secret).float().mean().item() * 100
            mse_val = F.mse_loss(stego, images).item()
            psnr = 20 * math.log10(1.0 / math.sqrt(mse_val + 1e-8))

            bit_accs.append(bit_acc)
            psnrs.append(psnr)

            pbar.set_postfix(BitAcc=f"{bit_acc:.2f}%", PSNR=f"{psnr:.2f}dB")

    return sum(bit_accs)/len(bit_accs), sum(psnrs)/len(psnrs)



# def main_worker(rank, world_size):
#     try:
#         # your existing code here
#         print(f"Starting training on GPU {rank}")
#         setup_distributed(rank, world_size)
#         device = torch.device(rank)
#         torch.cuda.set_device(device)

#         IMAGE_SIZE = 64
#         BATCH_SIZE = 128
#         NUM_WORKERS = 6
#         NUM_EPOCHS = 50
#         LEARNING_RATE = 1e-3
#         PAYLOAD_BITS = 32

#         train_transform = transforms.Compose([
#             transforms.RandomResizedCrop(IMAGE_SIZE),
#             transforms.RandomHorizontalFlip(),
#             transforms.ToTensor(),
#         ])
#         val_transform = transforms.Compose([
#             transforms.Resize(IMAGE_SIZE + 32),
#             transforms.CenterCrop(IMAGE_SIZE),
#             transforms.ToTensor(),
#         ])

#         tiny_root = './tiny-imagenet-200'  # CHANGE THIS PATH

#         train_dataset = ImageNetTrainDataset(tiny_root, transform=train_transform)
#         val_dataset = ImageNetValDataset(tiny_root, transform=val_transform)

#         train_sampler = DistributedSampler(train_dataset, num_replicas=world_size, rank=rank)
#         val_sampler = DistributedSampler(val_dataset, num_replicas=world_size, rank=rank, shuffle=False)

#         train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE // world_size, sampler=train_sampler,
#                                   shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
#         val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE // world_size, sampler=val_sampler,
#                                 shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

#         model = SteganographySystem(payload_bits=PAYLOAD_BITS).to(device)
#         model = DDP(model, device_ids=[rank])

#         optimizer_enc = optim.Adam(model.module.encoder.parameters(), lr=LEARNING_RATE)
#         optimizer_dec = optim.Adam(model.module.decoder.parameters(), lr=LEARNING_RATE * 2)
#         optimizer_disc = optim.Adam(model.module.discriminator.parameters(), lr=LEARNING_RATE * 0.5)

#         scheduler_enc = optim.lr_scheduler.CosineAnnealingLR(optimizer_enc, T_max=NUM_EPOCHS)
#         scheduler_dec = optim.lr_scheduler.CosineAnnealingLR(optimizer_dec, T_max=NUM_EPOCHS)
#         scheduler_disc = optim.lr_scheduler.CosineAnnealingLR(optimizer_disc, T_max=NUM_EPOCHS)

#         best_val_acc = 0.0
#         for epoch in range(NUM_EPOCHS):
#             train_sampler.set_epoch(epoch)

#             train_metrics, train_acc, train_psnr = train_one_epoch(epoch, model, train_loader,
#                                                                    optimizer_enc, optimizer_dec, optimizer_disc,
#                                                                    device)

#             val_acc, val_psnr = validate(epoch, model, val_loader, device)

#             scheduler_enc.step()
#             scheduler_dec.step()
#             scheduler_disc.step()

#             if rank == 0:
#                 print(f"Epoch {epoch+1} / {NUM_EPOCHS}")
#                 print(f"TRAIN: bit accuracy {train_acc:.2f}%, PSNR {train_psnr:.2f} dB")
#                 print(f"VAL:   bit accuracy {val_acc:.2f}%, PSNR {val_psnr:.2f} dB")
#                 if val_acc > best_val_acc:
#                     best_val_acc = val_acc
#                     torch.save({
#                         'epoch': epoch,
#                         'model_state_dict': model.module.state_dict(),
#                         'optimizer_enc': optimizer_enc.state_dict(),
#                         'optimizer_dec': optimizer_dec.state_dict(),
#                         'optimizer_disc': optimizer_disc.state_dict()
#                     }, 'best_steganography_tinyimagenet.pth')
#                     print("Saved new best model")

#             cleanup_distributed()
#     except Exception:
#             print(f"Exception in worker {rank}:")
#             traceback.print_exc()
#             raise  # re-raise for visibility or to properly clean up

def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Dataset paths and transforms
    TINYIMAGENET_ROOT = './tiny-imagenet-200'  # change accordingly
    IMAGE_SIZE = 64
    BATCH_SIZE = 128
    NUM_WORKERS = 6
    NUM_EPOCHS = 50
    LEARNING_RATE = 1e-3

    train_transform = transforms.Compose([
        transforms.RandomResizedCrop(IMAGE_SIZE),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
    ])
    val_transform = transforms.Compose([
        transforms.Resize(IMAGE_SIZE + 32),
        transforms.CenterCrop(IMAGE_SIZE),
        transforms.ToTensor(),
    ])

    # Use your previously defined Dataset classes for train/val here (or replace with ImageFolder if compatible)
    train_dataset = ImageNetTrainDataset(TINYIMAGENET_ROOT, transform=train_transform)
    val_dataset = ImageNetValDataset(TINYIMAGENET_ROOT, transform=val_transform)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)

    # Instantiate model
    model = SteganographySystem(payload_bits=32).to(device)

    # Optimizers and schedulers
    optimizer_enc = optim.Adam(model.encoder.parameters(), lr=LEARNING_RATE)
    optimizer_dec = optim.Adam(model.decoder.parameters(), lr=LEARNING_RATE * 2)
    optimizer_disc = optim.Adam(model.discriminator.parameters(), lr=LEARNING_RATE * 0.5)

    scheduler_enc = optim.lr_scheduler.CosineAnnealingLR(optimizer_enc, T_max=NUM_EPOCHS)
    scheduler_dec = optim.lr_scheduler.CosineAnnealingLR(optimizer_dec, T_max=NUM_EPOCHS)
    scheduler_disc = optim.lr_scheduler.CosineAnnealingLR(optimizer_disc, T_max=NUM_EPOCHS)

    best_val_acc = 0

    for epoch in range(NUM_EPOCHS):
        print(f"\n--- Epoch {epoch+1}/{NUM_EPOCHS} ---")

        train_metrics, train_acc, train_psnr = train_one_epoch(epoch, model, train_loader,
                                                               optimizer_enc, optimizer_dec, optimizer_disc, device)
        val_acc, val_psnr = validate(epoch, model, val_loader, device)

        scheduler_enc.step()
        scheduler_dec.step()
        scheduler_disc.step()

        print(f"Train Acc: {train_acc:.2f}% | PSNR: {train_psnr:.2f} dB")
        print(f" Val Acc: {val_acc:.2f}% | PSNR: {val_psnr:.2f} dB")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_enc': optimizer_enc.state_dict(),
                'optimizer_dec': optimizer_dec.state_dict(),
                'optimizer_disc': optimizer_disc.state_dict()
            }, 'best_steganography_tinyimagenet_singlegpu.pth')
            print(f"Saved best model checkpoint at epoch {epoch+1}")

if __name__ == "__main__":
  main()


--- Epoch 1/50 ---


Epoch 1 Validation: 100%|██████████| 79/79 [00:02<00:00, 29.17it/s, BitAcc=51.56%, PSNR=32.51dB]
/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:227: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


Train Acc: 50.01% | PSNR: 30.63 dB
 Val Acc: 50.14% | PSNR: 31.38 dB
Saved best model checkpoint at epoch 1

--- Epoch 2/50 ---


Epoch 2 Validation: 100%|██████████| 79/79 [00:02<00:00, 29.65it/s, BitAcc=50.00%, PSNR=41.72dB]


Train Acc: 50.00% | PSNR: 32.82 dB
 Val Acc: 49.91% | PSNR: 41.61 dB

--- Epoch 3/50 ---


...

Epoch 41 Validation: 100%|██████████| 79/79 [00:02<00:00, 29.83it/s, BitAcc=44.73%, PSNR=80.00dB]


Train Acc: 50.02% | PSNR: 79.99 dB
 Val Acc: 49.86% | PSNR: 80.00 dB

--- Epoch 42/50 ---


Epoch 42 Validation: 100%|██████████| 79/79 [00:02<00:00, 29.43it/s, BitAcc=50.98%, PSNR=80.00dB]


Train Acc: 49.95% | PSNR: 80.00 dB
 Val Acc: 50.01% | PSNR: 80.00 dB

--- Epoch 43/50 ---


Epoch 43 Training:  44%|████▍     | 346/782 [00:52<01:06,  6.55it/s, BitAcc=49.80%, Loss=14.20914, PSNR=80.00dB]